In [1]:
import openai
import pinecone
import json

from pinecone import Pinecone
from openai import OpenAI

In [2]:
import os

from dotenv import load_dotenv
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key = pinecone_api_key)
print("Pinecone client successfully configured.")
print(pinecone_api_key[:5])

openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key = openai_api_key)
print("OpenAI client successfully configured.")
print(openai_api_key[:5])

index_name = "faq-database"
index = pc.Index(index_name)

Pinecone client successfully configured.
pcsk_
OpenAI client successfully configured.
sk-pr


In [3]:
def embedding_model(query, openai_client, model="text-embedding-3-small"):
  response = openai_client.embeddings.create(
      model=model,
      input=query
  )

  embedding = response.data[0].embedding
  return embedding

In [4]:
system_prompt = {
                    "role": "system",
                    "content": f"""
                    You are a helpfull E-Commerce assistant helping customers with their general questions regarding policies and procedures when buying in our store.
                    Our store sells e-books and courses for IT professionals.
                    """,
                }

def prompt_builder(system_message, context):
  return system_message["content"].format(context)

In [5]:
def candidates_generation(query, openai_client, n_candidates=2):

  system_prompt = f"""
You are an AI based algorithm that has a task to generate
({n_candidates}) different versions of the user-generated question.
These questions will serve as candidates to retrieve relevant documents from vector database.
Questions should be short and to the point.
The output should be in the JSON format:
{{
    1: "candidate_one",
    2: "candidate_two",
    ...
    N: "candidate_five"
}}

Original question:
{query}
"""

  messages = [{"role": "system", "content": system_prompt}]

  response = openai_client.chat.completions.create(
      model="gpt-4o",
      messages=messages,
      max_tokens=1500,
      response_format={ "type": "json_object" }
    )

  response_content = response.choices[0].message.content
  response_type = json.loads(response_content)
  return response_type

In [6]:
def retrieve_faq_top_n(query_embedding, index, top_k=5):
    response = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="ns1"
    )
    results = []
    for res in response['matches']:
        results.append(res['metadata']['answer'])
    return results

In [7]:
def reciprocal_rank_fusion(results, k=60, top_n=5):
    ranked_docs = {}
    for docs in results:
        for i, doc in enumerate(docs):
            if doc not in ranked_docs:
                ranked_docs[doc] = 0
            ranked_docs[doc] += 1 / (k + i + 1)
    top_n_docs = [doc for doc, score in sorted(ranked_docs.items(), key=lambda item: item[1], reverse=True)[:top_n]]
    return top_n_docs

In [8]:
def combine_documents(retrieved_docs):
    return "\n\n".join(retrieved_docs)

In [9]:
def fusion_rag_chatbot(query, openai_client, index):

    candidates = candidates_generation(query, openai_client, n_candidates=4)

    relevant_docs = []
    for key, candidate in candidates.items():
      candidate_embedding = embedding_model(candidate, openai_client)
      best_match = retrieve_faq_top_n(candidate_embedding, index, top_k=5)
      relevant_docs.append(best_match)

    ranked_docs = reciprocal_rank_fusion(relevant_docs, k=60, top_n=5)

    context = combine_documents(ranked_docs)

    augmented_prompt = prompt_builder(system_prompt, context)

    messages = [{"role": "system","content": augmented_prompt},
                {"role": "user","content": query}]

    response = openai_client.chat.completions.create(
      model="gpt-4o",
      messages=messages,
      max_tokens=250,
      temperature=0
    )

    return response.choices[0].message.content

In [10]:
while True:
  query = input()
  response = fusion_rag_chatbot(query,client, index)
  print(f"User: {query}")
  print(f"Bot: {response}")
  exit_condition = input("Do you want to continue? (yes/no): ")
  if exit_condition.lower() != "yes":
    break

User: i had bought my product 40 days ago , can i return the item?
Bot: Our return policy allows for returns within 30 days of purchase. Since it has been 40 days since your purchase, it falls outside of our standard return window. However, if there are any issues with the product, such as defects or access problems, please let us know, and we will do our best to assist you.
